# Lab 2 — Design and Randomise the Injaz Uploader Experiment
# المختبر 2 — تصميم وعشوَنة تجربة رافع المستندات في إنجاز

**Module 2 · الوحدة 2 — Experiment Design and Randomisation / تصميم التجارب والعشوائية**
SDA-DSC-213 · Experimentation, A/B Testing and Causal Inference · SDAIA Academy

---

### Where Lab 1 left you

Lab 1 proved that a coin flip removes the selection-bias term. This lab does the coin
flip **for real**, on 182,113 eligible Injaz users, and then proves the flip actually
worked. A powerful analysis cannot rescue a broken design — this is the hour where
experiments are won or lost.

### What you will build

| # | Step | Minutes |
|---|---|---|
| 1 | The **design doc**: hypothesis, unit, OEC, guardrails, trigger, unit of analysis | 10 |
| 2 | `assign_variant` — deterministic **hash-based** assignment, then the library version | 8 |
| 3 | The **SRM** check: pass on clean data, then deliberately break it | 7 |
| 4 | The **balance table** over age, literacy, prior completions and region | 10 |
| 5 | The **interference audit** — two SUTVA risks and the design change each demands | 8 |
| 6 | The **frozen analysis plan** handed to Lab 4 | 7 |

### The vocabulary this lab installs (bilingual, once)

- **randomisation unit (وحدة العشوَنة)** — what gets independently coin-flipped.
- **OEC / Overall Evaluation Criterion (معيار التقييم الشامل)** — the single metric that decides the experiment.
- **guardrail metric (مقياس الحماية)** — a metric that must not get worse, whatever the OEC does.
- **trigger / exposure (التفعيل)** — the moment a unit actually experiences the difference.
- **SUTVA (فرضية ثبات قيمة معالجة الوحدة)** — no interference between units, one version of the treatment.
- **SRM / Sample Ratio Mismatch (عدم تطابق نسبة العينة)** — observed arm sizes differ from the planned split.
- **covariate balance (توازن المتغيرات المصاحبة)** — the arms look alike on everything measured before assignment.

> All code, variables and comments stay in **English**.


---
## Step 0 — The eligible pool

`injaz_users_pool.csv` is the population **before** the experiment: 200,000 users,
no outcomes. That absence is deliberate. Everything in this notebook must be decidable
without seeing a single outcome — that is what makes the design honest.


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `causal_utils` importable from anywhere under course_assets/
ROOT = Path.cwd()
while not (ROOT / "causal_utils").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from causal_utils import use_sdaia_style
use_sdaia_style()

RANDOM_SEED = 213          # fixed for every lab: reproducibility is graded
rng = np.random.default_rng(RANDOM_SEED)
print("Environment ready ·", ROOT)

In [ ]:
def needs(*values, msg="complete the TODO in the cell above, then re-run this cell"):
    """Guard for the start notebook: True (with a friendly note) if a TODO is unfinished."""
    if any(v is None for v in values):
        print(f"[not implemented yet] {msg}")
        return True
    return False

In [ ]:
import json
from causal_utils import (hash_assign, srm_check, standardised_mean_difference,
                          balance_table, NAVY, BLUE, ORANGE, TEAL, SLATE, MUTED, PANEL)

with open(DATA / "GROUND_TRUTH.json") as f:
    GROUND_TRUTH = json.load(f)

pool = pd.read_csv(DATA / "injaz_users_pool.csv")
eligible = pool.loc[pool["is_eligible"] == 1].copy().reset_index(drop=True)

print(f"pool rows      : {len(pool):,}")
print(f"eligible rows  : {len(eligible):,}   ({eligible.shape[0] / len(pool):.1%} of the pool)")
print(f"ground truth   : {GROUND_TRUTH['module2_user_pool']}")
print()
print(eligible[["user_id", "region", "age", "device", "prior_sessions",
                "prior_completion_rate", "digital_literacy_score"]].head())
print()
print("NOTE: there is no outcome column in this file, and there must not be. Every")
print("choice below is made blind to results — that is the definition of a design.")

---
## Step 1 *(10 min)* — The design doc

Fill every field. A field you cannot fill is a decision you have not made, and it will
be made for you later by whoever is looking at the dashboard.

**The rule for the randomisation unit:** *randomise at the coarsest unit interference
demands, and no coarser.* Coarser units contain spillover; they also destroy power.

Two sentences you must be able to defend out loud:

> *Why user and not session?*
> *Why is completion rate the OEC and not clicks on the upload button?*


In [ ]:
DESIGN_DOC = {
    "experiment_name": "injaz-guided-uploader-2026q2",

    # TODO: fill every field below. Strings, not placeholders.
    "decision": None,                              # the ship/hold question in one line
    "hypothesis": None,                            # what you expect, and by how much

    "randomisation_unit": None,                    # user? session? region?
    "randomisation_unit_justification": None,      # TWO sentences: why not session?

    "unit_of_analysis": None,
    "oec": None,                                   # exactly ONE primary metric
    "oec_justification": None,                     # why not "clicks on the button"?

    "guardrails": None,                            # a list of at least THREE
    "diagnostics": ["step-level drop-off", "trigger rate", "retries per upload"],
    "data_quality_checks": ["sample ratio (SRM)", "covariate balance", "missing-data rate"],

    "trigger_definition": None,                    # when does a user become exposed?
    "allocation": {"control": 0.5, "treatment": 0.5},
    "salt": "injaz-uploader-2026q2",
    "mde_absolute": 0.02,
    "alpha": 0.05,
    "power": 0.80,
    "practical_threshold": 0.01,
    "stopping_rule": "fixed horizon; no interim stopping for significance",
}

_required = ["decision", "hypothesis", "randomisation_unit",
             "randomisation_unit_justification", "unit_of_analysis", "oec",
             "oec_justification", "guardrails", "trigger_definition"]
_blank = [k for k in _required if DESIGN_DOC.get(k) is None]

if _blank:
    print(f"[not implemented yet] design doc fields still empty: {_blank}")
else:
    assert len(DESIGN_DOC["guardrails"]) >= 3, "you need at least three guardrails"
    assert DESIGN_DOC["randomisation_unit"] != DESIGN_DOC.get("trigger_definition")
    for k, v in DESIGN_DOC.items():
        print(f"{k}: {v}")
    print("\nDesign doc: all six required fields present.")

---
## Step 2 *(8 min)* — Build the assignment yourself, then check it

Assignment must be **deterministic, uniform, and independent across experiments**. The
professional pattern is hashing, not a per-request random draw:

```
hash(salt + ":" + unit_id) -> first 8 hex chars -> integer -> u in [0, 1) -> bucket
```

Three properties fall out for free:

1. **Deterministic** — the same user lands in the same arm on every request, with no
   stored state and no database lookup at serving time.
2. **Reproducible** — the analysis pipeline re-derives the assignment months later.
3. **Independent across experiments** — a different `salt` gives a fresh, uncorrelated
   bucketing, so concurrent experiments do not confound each other. Reusing a salt is
   how two teams silently ruin each other's results.

Build it yourself first. Then compare against `causal_utils.hash_assign` — they must
agree on **100%** of users, or one of you is wrong.


In [ ]:
import hashlib

def assign_variant(unit_id, experiment_salt, split=(0.5, 0.5),
                   arms=("control", "treatment")):
    """TODO: map ONE unit deterministically to an arm.

    Steps:
      1. digest = hashlib.md5(f"{experiment_salt}:{unit_id}".encode()).hexdigest()
      2. u = int(digest[:8], 16) / 0xFFFFFFFF          -> uniform in [0, 1)
      3. walk the cumulative shares in `split` and return the matching arm name

    Requirements you can self-check:
      - assign_variant(1, "s") must return the SAME arm every time it is called
      - across many ids the split must be ~50/50
      - a different salt must reshuffle the buckets

    Return None while unfinished.
    """
    assert abs(sum(split) - 1.0) < 1e-9, "split must sum to 1"
    # TODO: implement the three steps above
    return None


SALT = DESIGN_DOC["salt"]
_probe = assign_variant(1, SALT)

if not needs(_probe, msg="implement assign_variant"):
    mine = np.array([assign_variant(uid, SALT) for uid in eligible["user_id"].to_numpy()])
    library = hash_assign(eligible["user_id"].to_numpy(), SALT,
                          weights=(0.5, 0.5), arm_names=("control", "treatment"))
    agreement = (mine == library).mean()
    assert agreement == 1.0, (
        f"your assignment agrees with causal_utils on only {agreement:.2%} of users — "
        "check the hash (md5), the slice (first 8 hex chars) and the divisor (0xFFFFFFFF)")
    repeats = [np.array([assign_variant(uid, SALT)
                         for uid in eligible["user_id"].head(5_000)]) for _ in range(3)]
    assert all((repeats[0] == r).all() for r in repeats), "assignment is not deterministic"

    eligible["variant"] = library
    print(f"agreement with causal_utils.hash_assign : {agreement:.4%}")
    print(f"identical across 3 re-runs              : True")
    other = hash_assign(eligible["user_id"].head(50_000).to_numpy(), "injaz-autofill-2026q4")
    print(f"same arm under a DIFFERENT salt         : "
          f"{(other == library[:50_000]).mean():.3f}   (want ~0.50)")
    print()
    print(eligible["variant"].value_counts().to_string())
else:
    eligible["variant"] = None

### ✅ Checkpoint A — what you should be seeing by now

≈20 minutes in. Design doc complete, 182,113 users assigned, your hash agreeing with the
library on 100% of them, and the arms sitting close to 91,000 each.


In [ ]:
def checkpoint_a():
    if eligible["variant"].isna().all():
        print("✗ Step 2 incomplete — assign_variant is still a TODO.")
        return
    counts = eligible["variant"].value_counts()
    print(f"✓ design doc fields set   : {sum(v is not None for v in DESIGN_DOC.values())}/{len(DESIGN_DOC)}")
    print(f"✓ users assigned          : {len(eligible):,}")
    print(f"✓ control / treatment     : {counts.get('control', 0):,} / {counts.get('treatment', 0):,}")
    print(f"✓ treatment share         : {(eligible['variant'] == 'treatment').mean():.4f}   (planned 0.5000)")
    print()
    print("Next: prove the split is not just close, but statistically consistent with 50/50.")

checkpoint_a()

---
## Step 3 *(7 min)* — Sample Ratio Mismatch, and what a real bug looks like

You planned 50/50 and observed 91,167 / 90,946. Is that fine, or is it a bug?

The SRM check is a chi-square test of the observed arm counts against the planned split.
Two things about it that matter more than the arithmetic:

1. **α = 0.001, not 0.05.** With hundreds of thousands of units, harmless deviations trip
   a 5% threshold constantly, and a real SRM is usually catastrophic enough to blow past
   0.001 anyway.
2. **A failed SRM is not something you adjust for.** It means units were lost or
   misassigned *non-randomly*, so the comparison is no longer a randomised one. The
   correct response is to find the bug and rerun. **Never analyse an SRM-failing test.**

Build the check, confirm it passes — then deliberately drop 3% of the treatment arm,
the way a broken redirect or a logging race would, and watch it fail.


In [ ]:
from scipy import stats

def srm_check_manual(variant_series, expected_split, alpha=0.001):
    """TODO: chi-square goodness-of-fit on the arm counts.

    Return a dict with keys:
      arms, observed, expected, chi2, p_value, srm_detected, verdict

      chi2 = sum((observed - expected)**2 / expected)
      p    = stats.chi2.sf(chi2, df=len(arms) - 1)
      srm_detected = p < alpha        # alpha is 0.001 here, NOT 0.05 — see the text

    Self-check: the clean assignment must return srm_detected == False.
    """
    return None


expected_split = DESIGN_DOC["allocation"]
srm_clean = srm_check_manual(eligible["variant"], expected_split) \
    if eligible["variant"].notna().all() else None

if not needs(srm_clean, msg="implement srm_check_manual (and finish Step 2)"):
    assert srm_clean["srm_detected"] is False, \
        "a clean hash assignment must PASS the SRM check — check your expected counts"
    lib = srm_check(eligible["variant"], expected_split)
    assert abs(lib.chi2 - srm_clean["chi2"]) < 0.01, \
        f"your chi2 {srm_clean['chi2']} disagrees with causal_utils {lib.chi2:.3f}"
    print("A. Clean assignment")
    print(f"   {srm_clean}")
    print()
    print(lib)

    # --- TODO: inject the bug -------------------------------------------
    # Drop 3% of the TREATMENT rows at random (a redirect that silently loses
    # one arm) and re-run the check. It must now fail.
    g = np.random.default_rng(RANDOM_SEED)
    treatment_rows = eligible.index[eligible["variant"] == "treatment"]
    dropped = g.choice(treatment_rows, size=int(0.03 * len(treatment_rows)), replace=False)
    broken = eligible.drop(index=dropped)

    srm_broken = srm_check_manual(broken["variant"], expected_split)
    assert srm_broken["srm_detected"] is True, \
        "a 3% arm loss MUST be detected — if it is not, re-check your chi2"
    print("\nB. After a redirect silently drops 3% of the treatment arm")
    print(f"   {srm_broken}")
    print()
    print(srm_check(broken["variant"], expected_split))
else:
    srm_broken = None
    broken = None

In [ ]:
def plot_srm(clean, bad):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    for ax, res, title in [(axes[0], clean, "Clean assignment"),
                           (axes[1], bad, "3% of the treatment arm silently dropped")]:
        arms = res["arms"]
        x = np.arange(len(arms))
        ax.bar(x - 0.19, np.array(res["observed"]) / 1000, width=0.36,
               color=BLUE, label="observed")
        ax.bar(x + 0.19, np.array(res["expected"]) / 1000, width=0.36,
               color=MUTED, label="expected under the planned 50/50")
        ax.set_xticks(x)
        ax.set_xticklabels(arms)
        ax.set_ylabel("Users in arm (thousands)")
        verdict = "PASS" if not res["srm_detected"] else "INVALID"
        colour = TEAL if not res["srm_detected"] else "#C0392B"
        ax.set_title(f"{title}\nchi2 = {res['chi2']:,.1f}   p = {res['p_value']:.2g}   {verdict}",
                     color=colour, fontsize=11)
    axes[0].legend(loc="lower left")
    fig.suptitle("Sample Ratio Mismatch: the go / no-go gate before any analysis",
                 fontsize=13, fontweight="bold", color=NAVY)
    fig.text(0.5, -0.04,
             "Takeaway: the bars look identical to the eye in both panels. A 1.4 pp ratio drift "
             "is invisible\nand disqualifying — the arms are no longer a randomised comparison, "
             "so no analysis is valid.",
             ha="center", fontsize=10, color=SLATE,
             bbox=dict(boxstyle="round,pad=0.5", fc=PANEL, ec=MUTED, lw=0.8))
    plt.tight_layout()
    plt.show()


if not needs(srm_clean, srm_broken, msg="finish Step 3"):
    plot_srm(srm_clean, srm_broken)

---
## Step 4 *(10 min)* — Covariate balance, and the trap in this step

Randomisation balances covariates **in expectation** — not in any single draw. So you
check. The standard tool is the **standardised mean difference (الفرق المعياري في المتوسطات)**:

$$\text{SMD} = \frac{\bar{x}_{\text{treatment}} - \bar{x}_{\text{control}}}{\sqrt{(s^2_t + s^2_c)/2}}$$

Scale-free, so the same |SMD| < 0.1 threshold works for age in years, literacy in [0,1],
and a region indicator alike. **Categoricals must be one-hot expanded** — "region is
balanced" is not one number, each region is its own check.

> ### ⚠️ The trap in this step
> When a covariate comes out imbalanced, someone will want to **re-randomise until it
> looks better**. That is p-hacking the design: you are selecting an assignment on a
> property of the sample, which breaks the independence that made the assignment worth
> anything. **Randomise once, check, and adjust in analysis (Module 4).** Never re-roll.


In [ ]:
def smd(df, covariate, variant_col="variant", treated_label="treatment"):
    """TODO: standardised mean difference for ONE numeric covariate.

        SMD = (mean_treatment - mean_control) / sqrt((var_t + var_c) / 2)

    Use ddof=1 for the variances. Return 0.0 if the pooled sd is zero.
    """
    return None


def my_balance_table(df, covariates, variant_col="variant", threshold=0.10):
    """TODO: one row per covariate with columns covariate, smd, abs_smd, balanced.

    Numeric covariates -> one row.
    Categorical covariates -> ONE ROW PER LEVEL, using a 0/1 indicator
    (a single number cannot say whether 'region' is balanced).

    Sort by abs_smd descending so the worst covariate is the first row.
    """
    return None


COVARIATES = ["age", "digital_literacy_score", "prior_completion_rate", "prior_sessions",
              "device_age_years", "distance_km", "region", "device"]

bal = my_balance_table(eligible, COVARIATES) if eligible["variant"].notna().all() else None

if not needs(bal, msg="implement smd and my_balance_table"):
    lib_bal = balance_table(eligible, "variant", COVARIATES)
    assert len(bal) == len(lib_bal), (
        f"you produced {len(bal)} rows, causal_utils produced {len(lib_bal)} — "
        "did you expand the categorical covariates into one row per level?")
    assert abs(bal["abs_smd"].max() - lib_bal["abs_smd"].max()) < 1e-6, \
        "your worst |SMD| disagrees with causal_utils"
    assert bal["abs_smd"].max() < 0.10, "something is wrong: a clean hash split is balanced"
    print(bal.head(10).round(4).to_string(index=False))
    print(f"\ncovariates checked      : {len(bal)}")
    print(f"max |SMD|               : {bal['abs_smd'].max():.4f}")
    print(f"all below 0.10          : {bool(bal['balanced'].all())}")
    print(f"causal_utils max |SMD|  : {lib_bal['abs_smd'].max():.4f}   (must match)")

In [ ]:
def plot_balance(bal, threshold=0.10):
    top = bal.head(14).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9.5, 6))
    colours = [TEAL if b else ORANGE for b in top["balanced"]]
    ax.barh(top["covariate"], 100 * top["smd"], color=colours, height=0.62)
    for x in (-100 * threshold, 100 * threshold):
        ax.axvline(x, color=ORANGE, ls="--", lw=1.5)
    ax.axvline(0, color=MUTED, lw=1.1)
    ax.set_xlabel("Standardised mean difference (× 100; the |10| lines are the conventional threshold)")
    ax.set_title("Covariate balance after a single hash randomisation")
    ax.set_xlim(-12, 12)
    ax.annotate(
        f"Takeaway: the worst covariate is {100 * bal['abs_smd'].max():.1f} — "
        f"an eighth of the way to the\nthreshold. The coin flip worked. "
        "Do NOT re-roll to make it prettier.",
        xy=(0.02, 0.04), xycoords="axes fraction", fontsize=10, color=SLATE,
        bbox=dict(boxstyle="round,pad=0.5", fc=PANEL, ec=MUTED, lw=0.8))
    plt.tight_layout()
    plt.show()


if not needs(bal, msg="finish Step 4"):
    plot_balance(bal)

### The re-randomisation trap, demonstrated

Suppose you decide to re-roll the salt until *every* covariate has |SMD| < 0.005. It will
eventually happen. The cell below shows what you have actually done: you have chosen the
assignment based on the realised sample, which means the assignment is no longer
independent of the sample. The honest alternative is one salt, one roll, and covariate
adjustment in the analysis — which *also* buys you a tighter confidence interval, for free.


In [ ]:
sub = eligible.sample(20_000, random_state=RANDOM_SEED).copy()
rolls = []
for k in range(12):
    sub["v_try"] = hash_assign(sub["user_id"].to_numpy(), f"reroll-attempt-{k}")
    worst = balance_table(sub, "v_try", ["age", "digital_literacy_score",
                                         "prior_completion_rate"])["abs_smd"].max()
    rolls.append({"attempt": k, "worst_abs_smd": round(worst, 4)})

rolls = pd.DataFrame(rolls)
print(rolls.to_string(index=False))
print()
best = rolls.loc[rolls["worst_abs_smd"].idxmin()]
print(f"'Best' roll: attempt {int(best['attempt'])} with worst |SMD| = {best['worst_abs_smd']}")
print()
print("Every one of these twelve rolls is a valid randomisation. Picking the prettiest one")
print("is a choice made using the data, and it is no longer a coin flip. The 12 rolls are")
print("also the honest picture of chance imbalance: this is what 'balanced in expectation'")
print("looks like in practice, and why you check rather than assume.")

### ✅ Checkpoint B — what you should be seeing by now

≈37 minutes in. SRM passing on clean data and failing hard on the injected bug; a balance
table with a worst |SMD| around 0.009 and every row green.


In [ ]:
def checkpoint_b():
    if srm_clean is None or bal is None:
        print("✗ Steps 3-4 incomplete.")
        return
    print(f"✓ SRM, clean data        : chi2 = {srm_clean['chi2']:.2f}, "
          f"p = {srm_clean['p_value']:.3f}  -> {srm_clean['verdict']}")
    print(f"✓ SRM, 3% arm loss       : chi2 = {srm_broken['chi2']:.2f}, "
          f"p = {srm_broken['p_value']:.2g}  -> {srm_broken['verdict']}")
    print(f"✓ balance covariates     : {len(bal)} checked, max |SMD| = {bal['abs_smd'].max():.4f}")
    print(f"✓ all balanced (<0.10)   : {bool(bal['balanced'].all())}")
    print()
    print("The design is now defensible. Next: what could still break it, and freezing the plan.")

checkpoint_b()

---
## Step 5 *(8 min)* — Interference audit

Before you launch, ask one question: **can one user's treatment change another user's
outcome?** If yes, the user-level A/B test overstates its precision and possibly its
effect, and no amount of sample fixes it.

SUTVA fails in two ways worth naming:

- **Interference / spillover** — shared household devices, call-centre agents who learn
  the new flow and coach control users, capacity effects that free agents for everyone.
- **Multiple versions of the treatment** — "the uploader" is really three builds across
  OS versions, a hidden violation of the single-version assumption.

Containment designs, in order of increasing cost: cluster randomisation (household,
branch, region) → switchback / time-split → two-sided or ego-cluster designs.

Fill the audit below with **at least two** risks specific to this experiment.


In [ ]:
INTERFERENCE_AUDIT = [
    # TODO: at least TWO entries, each a dict with keys:
    #   risk, mechanism, severity, design_change
    #
    # Think about: shared household devices, call-centre agents who coach control
    # users, a fixed pool of appointment slots, three OS builds of "the" uploader.
]

if len(INTERFERENCE_AUDIT) < 2:
    print("[not implemented yet] add at least two SUTVA risks with mitigations")
else:
    for i, item in enumerate(INTERFERENCE_AUDIT, 1):
        print(f"{i}. {item['risk']}")
        print(f"   mechanism     : {item['mechanism']}")
        print(f"   severity      : {item['severity']}")
        print(f"   design change : {item['design_change']}\n")
    print(f"Audit complete: {len(INTERFERENCE_AUDIT)} SUTVA risks documented.")

---
## Step 6 *(7 min)* — Freeze the analysis plan

Pre-registration in miniature. Everything you must commit to **before** any outcome is
visible goes into an `AnalysisPlan`, which is then stamped and hashed. Put the hash in
the decision memo and a reader can verify nothing moved between design and analysis.

Metrics chosen after seeing data are not metrics — they are conclusions with a p-value
stapled on. This object is what Lab 4 opens.


In [ ]:
from causal_utils import AnalysisPlan, freeze_plan

# TODO: build the AnalysisPlan from DESIGN_DOC, freeze it, and write it to
# ROOT / "notebooks" / "lab2_frozen_plan.json".
#
#   plan = AnalysisPlan(experiment_name=..., decision=..., randomisation_unit=...,
#                       oec=..., guardrails=..., trigger_definition=...,
#                       unit_of_analysis=..., mde=..., alpha=..., power=...,
#                       correction_family=..., stopping_rule=...,
#                       practical_threshold=...)
#   plan = freeze_plan(plan)
plan = None

if not needs(plan, msg="build and freeze the AnalysisPlan"):
    assert plan.plan_hash is not None, "the plan must be frozen (hashed) before Lab 4"
    out_path = ROOT / "notebooks" / "lab2_frozen_plan.json"
    out_path.write_text(plan.to_json())
    print(f"\nwritten to {out_path.name}")
    print(plan.to_json())

---
## Step 7 — Grade yourself against the planted ground truth

`GROUND_TRUTH.json` records what the course generator actually produced. Compare your
eligible count, your realised allocation, and your SRM verdicts against it.


In [ ]:
M2 = GROUND_TRUTH["module2_user_pool"]
M4 = GROUND_TRUTH["module4_uploader_experiment"]

checks = [
    ("pool rows", len(pool), M2["rows"]),
    ("eligible rows", len(eligible), M2["eligible"]),
    ("treatment share (this randomisation)",
     round(float((eligible["variant"] == "treatment").mean()), 4)
     if eligible["variant"].notna().all() else float("nan"), 0.5),
    ("treatment share (course experiment file)",
     M4["allocation_treatment"], 0.5),
    ("treatment share (SRM-broken file)",
     M4["srm_broken_allocation"], 0.5),
]

print(f"{'quantity':<42}{'value':>14}{'target':>12}   verdict")
print("-" * 82)
for name, got, want in checks:
    if isinstance(got, int):
        ok = got == want
        print(f"{name:<42}{got:>14,}{want:>12,}   {'MATCH' if ok else 'CHECK'}")
    else:
        ok = abs(got - want) < 0.005 if got == got else False
        print(f"{name:<42}{got:>14.4f}{want:>12.4f}   "
              f"{'within 0.5 pp' if ok else 'SRM TERRITORY'}")

print()
print("Read the last two rows together. 0.5007 is a healthy experiment; 0.4928 is the")
print("SRM-broken file, and the difference between them is 0.8 percentage points.")
print("That is the entire margin between a result you can ship and one you must throw away.")

### Benchmark table for Module 2 (fill this in from your own run)

| Check | Result | Verdict |
|---|---|---|
| Randomisation determinism (3 re-runs) | | |
| SRM, intended 50/50 | | |
| SRM, injected 3% arm loss | | |
| Balance, max abs SMD | | |
| Design-doc completeness (6 fields) | | |
| Interference risks identified | | |


---
## What you now own

In [ ]:
print("""
TOOLKIT CONTRIBUTION — Lab 2
============================================================
Reusable artefacts you built and can lift into any project:

  assign_variant(unit_id, salt, split, arms)
      Deterministic, stateless, hash-based bucketing. Identical in the
      serving layer and the analysis pipeline, months apart. One salt per
      experiment keeps concurrent tests independent.

  srm_check_manual(variant, expected_split, alpha=0.001)
      The go / no-go gate. Run it BEFORE you compute any effect. A failing
      SRM is not a number to adjust, it is an experiment to rerun.

  smd(df, covariate) / my_balance_table(df, covariates)
      Scale-free balance checking with categoricals expanded per level.
      Run once, report, and adjust in analysis - never re-roll.

  DESIGN_DOC  +  INTERFERENCE_AUDIT
      The one-page design document and the SUTVA audit. These are the
      artefacts a reviewer reads first and a post-mortem reads last.

  lab2_frozen_plan.json  (AnalysisPlan, hashed)
      Pre-registration in miniature. Lab 4 opens this file BEFORE it opens
      the results, and the memo quotes its hash.

Carried forward:
  - Lab 3 turns DESIGN_DOC["mde_absolute"] into a sample size and a calendar.
  - Lab 4 opens the frozen plan, checks SRM again on the real results, and
    analyses the TRIGGERED users this design defined.
============================================================
""")

---
## Mini-exercises

### 1 · Quiz

1. When should you randomise by user rather than by session?
2. What is an OEC, and how many should an experiment have?
3. What does an SRM indicate, and what should you do about it?
4. Name a SUTVA-violating scenario and the containment design it demands.
5. Why hash instead of drawing a random number per request?

### 2 · Debugging exercise — the non-deterministic assignment

The cell below re-seeds the generator inside the loop, so the same user can land in
different arms across calls. Run it, then answer: what does the SRM check say, and why is
SRM *not* the diagnostic that catches this bug?

### 3 · Design critique

Three one-line designs. Name the flaw and the fix for each:

| # | Design | Flaw | Fix |
|---|---|---|---|
| a | "Session-level randomisation for a returning-user onboarding feature" | | |
| b | "Twelve primary metrics so we learn more" | | |
| c | "Region-level randomisation across our 13 regions, analysed with a standard z-test" | | |

### 4 · Discussion

- Your PM wants five "primary" metrics. What is the statistical cost, and what is your
  counter-proposal?
- A marketplace-style feature cannot be user-randomised safely. Walk through choosing
  between a cluster design and a switchback design.


In [ ]:
# --- Mini-exercise 2: the non-deterministic assignment --------------------
def assign_variant_BROKEN(unit_id, experiment_salt):
    """DELIBERATELY WRONG: a fresh random draw per call instead of a hash."""
    return np.random.default_rng().choice(["control", "treatment"])


# Run it, then write your answer in the markdown cell above.
sample_ids = eligible["user_id"].head(20_000).to_numpy()
run_a = np.array([assign_variant_BROKEN(u, SALT) for u in sample_ids])
run_b = np.array([assign_variant_BROKEN(u, SALT) for u in sample_ids])

print(f"same arm across two runs : {(run_a == run_b).mean():.3f}   (a correct hash gives 1.000)")
print()
print("And the SRM check on run A:")
print(srm_check(pd.Series(run_a), {"control": 0.5, "treatment": 0.5}))
print()
print("SRM PASSES. That is the lesson: SRM detects units LOST from an arm, not units")
print("shuffled BETWEEN arms. The diagnostic for non-determinism is re-running the")
print("assignment and comparing - which is why the determinism check in Step 2 exists.")